<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la ruta de rescate docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: ATE con IC 95 % y artefacto DML guardado, o el número del solution marcado como referencia.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la ruta de rescate docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: comparar PLR e IRM, learners y sensibilidad completa.
<!-- OPS:MAT-02:fin -->

# L10 · DML: PLR, IRM y soporte común

**S10 · Lectura: Cap. 11 · 40–55 min**

## La pregunta

¿Cuánto cambia el resultado y con qué incertidumbre?

## Qué harás

Comparar PLR e IRM y revisar overlap, ESS y trimming.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** ATE, intervalo, ESS relativo y soporte antes/después del recorte.

> **Idea clave:** Un número causal necesita un intervalo y una población donde la comparación sea posible.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor, ExtraTreesClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import RidgeCV, LogisticRegression
rng = setup()
sc = load_scenario("synthetic/inference/ihdp_medium"); sc.card()
X, Y, T = sc.xyt("train")
print("n =", len(Y), "· p =", X.shape[1])

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. Dos formas de estimar el mismo ATE (§11.1–11.3)
Ambas usan **nuisance con cross-fitting** (Cap. 10), pero resumen distinto:

- **PLR** (modelo parcialmente lineal): $\hat\theta=\dfrac{\sum \tilde D\,\tilde Y}{\sum \tilde D^2}$ con $\tilde Y=Y-\hat m(X)$, $\tilde D=D-\hat e(X)$. Supone un efecto **constante**.
- **IRM** (modelo de regresión interactivo, score AIPW): $\hat\tau=\dfrac1n\sum\Big[\hat\mu_1-\hat\mu_0+\dfrac{D(Y-\hat\mu_1)}{\hat e}-\dfrac{(1-D)(Y-\hat\mu_0)}{1-\hat e}\Big]$. Permite efecto **heterogéneo** y promedia.

Primero, el cross-fitting de los cuatro nuisance: $\hat\mu_1,\hat\mu_0$ (resultado por brazo), $\hat m=E[Y|X]$ y $\hat e$.

In [ ]:
def cf_nuisance(X, y, t, make_reg, make_clf, K=5, seed=SEED):
    n = len(y); o = {k: np.zeros(n) for k in ("mu0", "mu1", "m", "e")}
    for te in kfold(n, K, seed):
        tr = np.setdiff1d(np.arange(n), te)
        tr1, tr0 = tr[t[tr] == 1], tr[t[tr] == 0]
        o["mu1"][te] = make_reg().fit(X[tr1], y[tr1]).predict(X[te])
        o["mu0"][te] = make_reg().fit(X[tr0], y[tr0]).predict(X[te])
        o["m"][te]   = make_reg().fit(X[tr],  y[tr]).predict(X[te])
        o["e"][te]   = make_clf().fit(X[tr],  t[tr]).predict_proba(X[te])[:, 1]
    return o

def plr(y, t, nu):
    rd, ry = t - nu["e"], y - nu["m"]
    th = (rd * ry).sum() / (rd * rd).sum()
    psi = (ry - th * rd) * rd / np.mean(rd ** 2)
    return th, np.sqrt(np.mean(psi ** 2) / len(y))

def irm(y, t, nu, clip=0.01, mask=None):
    e = np.clip(nu["e"], clip, 1 - clip)
    psi = nu["mu1"] - nu["mu0"] + t * (y - nu["mu1"]) / e - (1 - t) * (y - nu["mu0"]) / (1 - e)
    if mask is not None: psi = psi[mask]
    return psi.mean(), psi.std(ddof=1) / np.sqrt(len(psi))

ET_R = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=5, max_features=0.8, random_state=0)
ET_C = lambda: ExtraTreesClassifier(n_estimators=40, min_samples_leaf=15, max_features=0.8, random_state=0)
nu = cf_nuisance(X, Y, T, ET_R, ET_C, K=5)
print("PLR :", fmt_ci(*plr(Y, T, nu)))
print("IRM :", fmt_ci(*irm(Y, T, nu)))
print("dif. de medias (ingenuo):", fmt_ci(*diff_in_means(Y, T)))

**Mira el resultado.** ¿Por qué PLR e IRM no tienen por qué coincidir exactamente aunque usen los mismos nuisance?

**Respuesta:**

PLR impone un único θ (promedio ponderado por $e(1-e)$ de los efectos individuales si el efecto es heterogéneo); IRM promedia el score AIPW sobre **todas** las unidades (el ATE puro). Si el efecto varía con $X$, apuntan a estimandos algo distintos.



## 2. Diagnóstico de solapamiento (§11.4)
El IRM divide por $\hat e$ y $1-\hat e$. Si hay unidades con $\hat e\approx 0$ o $1$, el estimador tiene **varianza enorme**. Dos diagnósticos:

1. **Fracción de $\hat e$ fuera de [0,05; 0,95]** (regla del capítulo: más de 5 % ⇒ alerta).
2. **ESS relativo de Kish** de los pesos IPW, $\;\mathrm{ESS}/n=\dfrac{(\sum w)^2}{n\sum w^2}\in(0,1]$: qué fracción de la muestra "vale" tras ponderar.

### Completa tú
`ess_relative(w)` y `recorte(e, lo, hi)` (devuelve una máscara booleana con las unidades que **se conservan**).

In [ ]:
# ✏️ Completa tú — ess_relative y recorte
# Pista: Si todos los pesos son iguales, el ESS relativo es 1.
def ess_relative(w):
    """ESS relativo de Kish: (Σw)² / (n Σw²)."""
    raise NotImplementedError("completa ess_relative")

def recorte(e, lo=0.05, hi=0.95):
    """Máscara booleana: True si la unidad se conserva (lo ≤ ê ≤ hi)."""
    raise NotImplementedError("completa recorte")

In [ ]:
lab_check("ess_relative", ess_relative, [((np.ones(10),), 1.0, 1e-9), ((np.array([1, 1, 1, 9.]),), 144 / 84 / 4, 1e-9)])
lab_check("recorte", lambda e: recorte(e).tolist(), [((np.array([0.01, 0.5, 0.97]),), [False, True, False], 0)])

In [ ]:
def w_ipw(t, e): return t / e + (1 - t) / (1 - e)
e_hat = nu["e"]
fuera = 1 - recorte(e_hat).mean()
print(f"fracción de ê fuera de [0,05; 0,95]: {fuera:.3f}   (alerta si > 0,05)")
print(f"ESS relativo de los pesos IPW: {ess_relative(w_ipw(T, np.clip(e_hat, .01, .99))):.3f}")
plt.hist(e_hat[T == 1], bins=30, alpha=.6, label="tratados"); plt.hist(e_hat[T == 0], bins=30, alpha=.6, label="controles")
plt.axvline(.05, c="r", ls="--"); plt.axvline(.95, c="r", ls="--"); plt.xlabel("ê(X)"); plt.legend(); plt.show()

## 3. La señal de alerta: solapamiento deficiente
En IHDP-medium el solapamiento es razonable. Para **ver** la alerta construimos un mundo con la misma estructura pero asignación casi determinista ($\text{logit}\,e=4\,x_0$), con **verdad conocida**: efecto heterogéneo $\tau(x)=1+x_1$ (ATE = 1).

In [ ]:
def mundo(n=3000, fuerza=1.0, rng=None):
    rng = rng or np.random.default_rng(3)
    X_ = rng.normal(size=(n, 4)); e_ = 1 / (1 + np.exp(-fuerza * X_[:, 0] * 4))
    t_ = rng.binomial(1, e_); tau_ = 1 + X_[:, 1] + 0.8 * X_[:, 0]          # efecto mayor donde e es alto
    y_ = 0.5 * X_[:, 0] + np.sin(X_[:, 2]) + t_ * tau_ + rng.normal(size=n)
    return X_, y_, t_, tau_
Xw, Yw, Tw, tauw = mundo()
nuw = cf_nuisance(Xw, Yw, Tw, ET_R, ET_C, K=5)
ew = nuw["e"]; keep = recorte(ew)
print(f"fracción de ê fuera de [0,05; 0,95]: {1 - keep.mean():.2f}  ← alerta")
print(f"ESS relativo (sin recortar): {ess_relative(w_ipw(Tw, np.clip(ew, .01, .99))):.3f} · tras recortar: {ess_relative(w_ipw(Tw[keep], ew[keep])):.3f}")
a, b = irm(Yw, Tw, nuw), irm(Yw, Tw, nuw, mask=keep)
print("IRM sin recortar :", fmt_ci(*a), f"  (n = {len(Yw)})")
print("IRM con recorte  :", fmt_ci(*b), f"  (n = {keep.sum()})")
# EVALUATION-CELL
print(f"verdad: ATE de todos = {tauw.mean():.3f} · ATE de los conservados = {tauw[keep].mean():.3f}")

**Mira el resultado.** ¿A qué ATE apunta el estimador recortado? ¿Qué debes reportar?

**Respuesta:**

Tras recortar, el estimando es el efecto **en la región de solapamiento**, que puede diferir del ATE de toda la población (aquí el efecto crece con $x_0$, justo donde se pierde solapamiento). Se reportan **ambos** resultados (con y sin recorte), el porcentaje de unidades descartadas y el ESS: el recorte reduce varianza pero **cambia la pregunta**. Fíjate además en que, con este solapamiento, **ningún** IC incluye el ATE verdadero: con propensities extremos el error estándar subestima la incertidumbre real. Un IC estrecho no certifica nada si el solapamiento falla.



## 4. Criterios del curso para el solapamiento (conflicto C8)
El capítulo usa ê ∉ [0,05; 0,95] en más de 5 %. El sílabo propone además ESS relativo con umbrales 0,30 (revisar) y 0,15 (grave). Aplicamos **ambos** a los dos mundos.

In [ ]:
def semaforo(e, t):
    f = 1 - recorte(e).mean(); ess = ess_relative(w_ipw(t, np.clip(e, .01, .99)))
    regla_libro = "ALERTA" if f > .05 else "ok"
    regla_silabo = "grave" if ess < .15 else ("revisar" if ess < .30 else "ok")
    return pd.Series({"fuera de [.05,.95]": round(f, 3), "libro": regla_libro, "ESS/n": round(ess, 3), "sílabo": regla_silabo})
show(pd.DataFrame({"IHDP-medium": semaforo(nu["e"], T), "mundo con mal solapamiento": semaforo(ew, Tw)}))

**Mira el resultado.** En el mundo con mal solapamiento las dos reglas no dicen lo mismo. ¿Cuál es más exigente y por qué conviene reportar las dos?

**Respuesta:**

La regla del libro (fracción de ê extremos) salta con 21 %, mientras que el ESS relativo (≈ 0,50) queda por encima de los umbrales del sílabo (0,30 / 0,15). Miden cosas distintas: una cuenta unidades con propensity casi 0 o 1; la otra resume cuánta información se pierde por la dispersión de los pesos. Se reportan ambas y se usa la **más severa** para decidir (aquí: alertar).



## 5. Sensibilidad al learner y a K (§11.5–11.6)
Un resultado que cambia mucho con el learner o con K es una señal de inestabilidad. Comparamos tres learners y tres valores de K (tres semillas de partición cada uno).

In [ ]:
learners = {
    "ExtraTrees": (ET_R, ET_C),
    "Boosting": (lambda: HistGradientBoostingRegressor(max_iter=80, learning_rate=.05, random_state=0),
                 lambda: HistGradientBoostingClassifier(max_iter=60, learning_rate=.05, random_state=0)),
    "Lineal": (lambda: RidgeCV(alphas=np.logspace(-2, 2, 9)), lambda: LogisticRegression(C=1.0, max_iter=500)),
}
filas = []
for nom, (mr, mc) in learners.items():
    n_ = cf_nuisance(X, Y, T, mr, mc, K=5)
    filas.append((nom, *irm(Y, T, n_), *plr(Y, T, n_)))
tab = pd.DataFrame(filas, columns=["learner", "IRM", "se IRM", "PLR", "se PLR"]).set_index("learner")
show(tab.round(3))

In [ ]:
filas = []
for K in (2, 5, 10):
    est = [irm(Y, T, cf_nuisance(X, Y, T, ET_R, ET_C, K=K, seed=s))[0] for s in (1, 2, 3)]
    filas.append((K, np.mean(est), np.ptp(est)))
show(pd.DataFrame(filas, columns=["K", "IRM (media de 3 semillas)", "rango entre semillas"]).set_index("K").round(3))

In [ ]:
# EVALUATION-CELL
print("ATE verdadero:", round(sc.true_ate, 3))
print("Error absoluto de cada learner (IRM):", {k: round(abs(v - sc.true_ate), 3) for k, v in tab["IRM"].items()})

**Mira el resultado.** Si los tres learners dan valores parecidos y el rango entre semillas es pequeño, ¿qué puedes decir? ¿Y si no?

**Respuesta:**

Que el resultado **no depende de una elección arbitraria** (evidencia de estabilidad, no de verdad). Si cambia mucho, hay que reportarlo como incertidumbre adicional y revisar el solapamiento o la calidad de los nuisance.



## 6. Doble robustez (§11.7, ejercicio)
El score AIPW es consistente si **al menos uno** de los dos modelos (resultado o propensity) es correcto. Lo comprobamos con nuisance **conocidos** que estropeamos a propósito: $\mu$ mala = constante 0; $e$ mala = 0,5 constante. Verdad: ATE = 1.

In [ ]:
N_REP = R(30, 300)
def una(rng, n=3000):
    X_ = rng.normal(size=(n, 3)); e_ = 1 / (1 + np.exp(-(1.0 * X_[:, 0] + 0.5 * X_[:, 1])))
    t_ = rng.binomial(1, e_); mu0_ = 2 + X_[:, 0] ** 2 + X_[:, 1]
    y_ = mu0_ + 1.0 * t_ + rng.normal(size=n)
    return X_, y_, t_, e_, mu0_
def aipw(y, t, e, mu0, mu1):
    return np.mean(mu1 - mu0 + t * (y - mu1) / e - (1 - t) * (y - mu0) / (1 - e))
res = {k: [] for k in ("ambos buenos", "resultado malo · e bueno", "resultado bueno · e malo", "ambos malos")}
r_ = np.random.default_rng(5)
for _ in range(N_REP):
    X_, y_, t_, e_, mu0_ = una(r_)
    for k in res:
        e_use = e_ if k in ("ambos buenos", "resultado malo · e bueno") else np.full(len(y_), .5)
        m0, m1 = (mu0_, mu0_ + 1.0) if k in ("ambos buenos", "resultado bueno · e malo") else (np.zeros(len(y_)), np.zeros(len(y_)))
        res[k].append(aipw(y_, t_, e_use, m0, m1))
show(pd.DataFrame({"sesgo": {k: np.mean(v) - 1 for k, v in res.items()}, "desv. est.": {k: np.std(v) for k, v in res.items()}}).round(3))

**Mira el resultado.** ¿En qué fila(s) aparece sesgo y qué dice eso sobre dónde conviene invertir esfuerzo?

**Respuesta:**

Solo cuando **ambos** modelos son malos. Basta un nuisance correcto para evitar el sesgo asintótico; por eso conviene modelar *los dos* con flexibilidad y comprobar la calidad de cada uno, no depender de uno solo. (La doble robustez protege el sesgo, no la varianza: con propensity malo el error estándar puede crecer.)



## 7. Supuesto y bitácora

In [ ]:
e_ihdp = nu["e"]; f0 = 1 - recorte(e_ihdp).mean(); ess0 = ess_relative(w_ipw(T, np.clip(e_ihdp, .01, .99)))
est, se = irm(Y, T, nu)
fila = bitacora(
    pregunta="¿Cuál es el ATE en IHDP-medium con IRM y cross-fitting, y es creíble el solapamiento?",
    metodo="DML-IRM (AIPW) con ExtraTrees, K=5; PLR como contraste",
    estimando="ATE",
    numero_ic=fmt_ci(est, se),
    supuesto=supuesto("observable", "medido", f"solapamiento: {100 * f0:.1f}% de ê fuera de [0,05; 0,95]; ESS/n = {ess0:.2f}"),
    decision="continuar" if (f0 <= .05 and ess0 >= .30) else "alertar", lab="L10")

---
## Qué queda para el libro
Quedan en el libro: los listados con `DoubleML` y `econml` (no se instalan aquí; el lab usa cross-fitting propio), la teoría de la condición de tasa y las refutaciones del capítulo.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*